In [1]:
import os
import torch
from torch import nn
import numpy as np
from transformers import AutoTokenizer

c:\Users\tykhi\Documents\University\Semester 5\AI Lab\Transformer project\transformer-from-scratch-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
# Transformer parameters
model_dimmension = 512
num_heads =  8
num_layers = 12
ff_network_dimention = 2048
max_sequence_length = 100
dropout = 0.1

# Training parameters
leerning_rate = 0.0001
epochs_number = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

[INFO] Using device: cpu


In [35]:
# Data loading
train_data = np.load("train.npy")
print(train_data)
print(len(train_data))
train_data = torch.from_numpy(train_data).long()
validation_data = np.load("val.npy")
validation_data = torch.from_numpy(validation_data).long()
# test_data = np.load("test.npy")

[   50  2978    77   461   357    11   635  7993  1143   355   911 25786
   461     8   318   257  7404   287  1081  3301 36486  5665    11  1081
  3301  5665    11  9375  1228  3418    11   978  2865    89 22783    11
  4068    13  1629   262  4793 21649    11   663  3265   373 20416    11
   287  6073  4172    13   198   198 19927   220   198   198 16979  4817
  4113   287  9375  1228  3418    33   964  1636  3250   318   257  1171
  3952 22765   287   262   968 37490   268  1989   286   262  8111 48114
   286 34612  2402 42521   287  5366    12 14197  3576    11  4492    13
   632   318   262  4067   810   262   350  2645  7590 15449   262 22185
  1636  7590    13   198   198  3791 37490   268   198    47  5558   290
  1280  9029   287   262  8111 48114   286 34612]
128


In [30]:
print(train_data.size())
print(validation_data.shape)

torch.Size([128])
torch.Size([1])


In [18]:
class PositionalEmbedding(nn.Module):
    def __init__(self, model_dimention, max_sequence_length):
        super(PositionalEmbedding, self).__init__()
        positional_embedding = torch.zeros(max_sequence_length, model_dimention)

        position = torch.arange(start=0, end=max_sequence_length, dtype=torch.float)
        position = position.unsqueeze(1)

        # Frequency components of the sine waves
        scale = torch.exp(torch.arange(0, model_dimention, 2).float() * -(np.log(10000.0) / model_dimention))

        positional_embedding[:, 0::2] = torch.sin(position * scale)
        positional_embedding[:, 1::2] = torch.cos(position * scale)
        self.positional_embedding = positional_embedding.unsqueeze(0)

        self.register_buffer("pos_embed", self.positional_embedding)

    def forward(self, embedded_tokens):
        return embedded_tokens + self.positional_embedding[:,:embedded_tokens.size(1)]

In [19]:
class MaskedMultiheadAttention(nn.Module):
    def __init__(self, model_dimention, heads_number):
        super(MaskedMultiheadAttention, self).__init__()
        self.model_dimention = model_dimention
        self.heads_number = heads_number
        self.keys_dimention = model_dimention // heads_number

        self.query_weigth = nn.Linear(model_dimention, model_dimention)
        self.key_weigth = nn.Linear(model_dimention, model_dimention)
        self.value_weigth = nn.Linear(model_dimention, model_dimention)
        self.output_weigth = nn.Linear(model_dimention, model_dimention)

    def forward(self, query, key, value, mask):
        Q = self.reshape_matrix(self.query_weigth(query))
        K = self.reshape_matrix(self.key_weigth(key))
        V = self.reshape_matrix(self.value_weigth(value))
        attention_output = self.attention_calculation(query=Q, key=K, value=V, mask=mask)

        batch_size, _, seq_length, d_k = attention_output.size()
        combined_matrix = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.model_dimention)
        output = self.output_weigth(combined_matrix)
        return output

    def reshape_matrix(self, matrix):
        batch_size, sequence_length, model_dimention = matrix.size()
        return matrix.view(batch_size, sequence_length, self.heads_number, self.keys_dimention).transpose(1, 2)

    def attention_calculation(self, query, key, value, mask):
        attnention_scores = torch.matmul(query, key.transpose(-2, -1)) / np.sqrt(self.keys_dimention)
        attnention_scores = attnention_scores.masked_fill(mask==0, -1e9)
        
        attnention_probabilities = torch.softmax(attnention_scores, dim=-1)
        return torch.matmul(attnention_probabilities, value)

In [20]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, model_dimention, ff_network_dimention):
        super(FeedForwardNetwork, self).__init__()
        self.layer1 = nn.Linear(model_dimention, ff_network_dimention)
        self.layer2 = nn.Linear(model_dimention, ff_network_dimention)
        self.relu = nn.ReLU()

    def forward(self, input_matrix):
        self.layer2(self.relu(self.layer1(input_matrix)))

In [21]:
class DecoderBlock(nn.Module):
    def __init__(self, model_dimention, heads_number, ff_network_dimention, dropout):
        super(DecoderBlock, self).__init__()
        self.attention = MaskedMultiheadAttention(model_dimention, heads_number)
        self.feed_forward = FeedForwardNetwork(model_dimention, ff_network_dimention)
        self.norm1 = nn.LayerNorm(model_dimention)
        self.norm2 = nn.LayerNorm(model_dimention)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_matrix, mask):
        attention_output = self.attention.forward(input_matrix, input_matrix, input_matrix, mask)
        normalised_attention_output = self.norm1(input_matrix + self.dropout(attention_output))
        ff_output = self.feed_forward.forward(normalised_attention_output)
        output = self.norm2(input_matrix + self.dropout(ff_output))
        return output

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, model_dimention, max_sequence_length, heads_number, ff_network_dimention, dropout, num_layers):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, model_dimention)
        self.positional_embedding = PositionalEmbedding(model_dimention, max_sequence_length)
        self.multi_head = MaskedMultiheadAttention(model_dimention, heads_number)
        self.feed_forward = FeedForwardNetwork(model_dimention, ff_network_dimention)
        self.dropout = nn.Dropout(dropout)

        self.decoder_layers = nn.ModuleList([DecoderBlock(model_dimention, heads_number, ff_network_dimention, dropout) for _ in range(num_layers)])
        self.connected_layer = nn.Linear(model_dimention, vocab_size)

    def forward(self, target):
        mask = (target != 0)
        _, sequence_length = target.shape()
        nopeak_mask = torch.tril(torch.ones(sequence_length, sequence_length, device=device, dtype=torch.bool))
        attention_mask = mask[:, None, None, :] & nopeak_mask[None, None, :, :]

        dec_output = self.dropout(self.positional_embedding(self.embedding(target)))
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, attention_mask)

        output = self.connected_layer(dec_output)
        return output

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
vocab_size = len(tokenizer.get_vocab())

losses_array = []
accuracy_array = []

model_dimmension = 512
num_heads =  8
num_layers = 12
ff_network_dimention = 2048
max_sequence_length = 100
dropout = 0.1

transformer = Transformer(vocab_size, model_dimmension, max_sequence_length, num_heads, ff_network_dimention, dropout, num_layers)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)


for epoch in range(epochs_number):
    transformer.train()
    optimizer.zero_grad()
    output = transformer.forward(train_data[:, :-1])
    loss = loss_function(output.contiguous().reshape(-1, vocab_size), train_data[:, 1:].reshape(-1))
    
    loss.backward()
    optimizer.step()
    print(f"[INFO] Epoch: {epoch+1}, Loss: {loss}")

    transformer.eval()

    with torch.no_grad():
        val_output = transformer(validation_data[:, :-1])
        val_loss = loss_function(val_output.contiguous().view(-1, vocab_size), validation_data[:, 1:].contiguous().view(-1))
        print(f"Validation Loss: {val_loss.item()}")

RuntimeError: The size of tensor a (128) must match the size of tensor b (100) at non-singleton dimension 1